# YAXCHÉ 🌳🔥: Simulación Espacio-Temporal de Incendios en Guatemala
### Pipeline de Referencia (Baseline Secuencial) para Aceleración en Clúster Kabré (CeNAT)

> **Origen del Proyecto:** En la cosmovisión maya, la **Yaxché (la Ceiba Pentandra)** es el *Árbol de la Vida* que conecta las estrellas con la tierra viva. **YAXCHÉ** es un proyecto de modelado computacional que acopla datos de observación de la Tierra (**NASA SRTM, FIRMS, NOAA GFS**) con simulación física acelerada en GPU para anticipar la propagación de frentes térmicos y proteger el patrimonio forestal de Guatemala.

## 1. Importación de Módulos y Dependencias

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import time
import numpy as np
import matplotlib.pyplot as plt

from src.terrain import TerrainGrid
from src.wind import WindField
from src.simulation import FireSimulation, STATE_UNBURNED, STATE_BURNING, STATE_BURNED, run_benchmark
from src.visualization import plot_simulation_state

%matplotlib inline
print("Módulos de YAXCHÉ cargados con éxito.")

## 2. Generación y Visualización del Terreno (Malla Espacial de Guatemala)

Generamos una malla topográfica con las características orográficas de Guatemala:
- **Norte (Petén):** Tierras bajas con alta densidad de biomasa forestal.
- **Centro / Occidente:** Cordillera volcánica (Sierra Madre y Cuchumatanes) con fuertes pendientes.
- **Sur:** Planicie costera hacia el Océano Pacífico.

In [ ]:
# Dimensiones de prueba (ej. 300x300 celdas a 100m de resolución)
filas, cols = 300, 300
terreno = TerrainGrid.create_synthetic_guatemala(rows=filas, cols=cols, cell_size_meters=100.0)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
im0 = ax[0].imshow(terreno.elevation, cmap='terrain')
ax[0].set_title("Modelo Digital de Elevación (DEM Sintético)")
plt.colorbar(im0, ax=ax[0], label="Metros s.n.m.")

im1 = ax[1].imshow(terreno.fuel, cmap='YlGn')
ax[1].set_title("Mapa de Densidad de Combustible (Biomasa)")
plt.colorbar(im1, ax=ax[1], label="Coeficiente [0.0 - 1.0]")
plt.show()

## 3. Configuración del Campo Atmosférico y Puntos de Ignición

Configuramos un viento de 15 m/s soplando hacia el Noreste (45°), simulando vientos alisios típicos de la temporada seca.

In [ ]:
# Campo de viento uniforme (velocidad 12 m/s, dirección 45°)
viento = WindField.uniform(shape=(filas, cols), speed_ms=12.0, direction_degrees=45.0)

# Inicializar simulador
sim = FireSimulation(terrain=terreno, wind=viento)

# Iniciar 2 focos térmicos: uno en Petén (norte) y otro en las faldas volcánicas (sur-centro)
sim.ignite(row=int(filas * 0.8), col=int(cols * 0.5))  # Foco Petén
sim.ignite(row=int(filas * 0.35), col=int(cols * 0.45)) # Foco Volcánico

plot_simulation_state(sim, title="YAXCHÉ - Estado Inicial (Ignición)")
plt.show()

## 4. Ejecución del Avance Temporal (Física de Propagación)

Avanzamos la simulación varios pasos temporales secuenciales. Notarás cómo el fuego se deforma hacia el Noreste impulsado por el viento y acelera al subir pendientes.

In [ ]:
pasos = 15
print(f"Ejecutando {pasos} pasos temporales secuenciales...")
t_inicio = time.perf_counter()
historial = sim.run(steps=pasos)
t_total = time.perf_counter() - t_inicio

print(f"Completado en {t_total:.4f} segundos ({t_total/pasos:.4f} s/paso).")
plot_simulation_state(sim, title=f"YAXCHÉ - Frente Térmico tras {pasos} pasos")
plt.show()

## 5. Benchmark Inicial y Demostración del Cuello de Botella

Evaluamos cómo escala el tiempo de cálculo en CPU monohilo conforme aumentamos el tamaño de la malla geográfica. **Esta tabla es la justificación cuantitativa para el clúster Kabré.**

In [ ]:
# Ejecutar benchmark de referencia en diferentes tamaños de malla
resultados = run_benchmark(sizes=[100, 200, 300], steps_per_size=5)

tamanios = [r[0] for r in resultados]
tiempos = [r[2] for r in resultados]

plt.figure(figsize=(8, 4), dpi=100)
plt.plot(tamanios, tiempos, 'o-', color='crimson', linewidth=2, label='CPU Monohilo (Python Puro)')
plt.title("Cuello de Botella Computacional en CPU Secuencial")
plt.xlabel("Dimensión de Malla (N x N)")
plt.ylabel("Tiempo de Cálculo (s)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

## 6. Siguientes Pasos en el Taller de HPC (Kabré)

1. **Mañana (CPU):** Reemplazar los bucles anidados en `FireSimulation.step_sequential()` por funciones compiladas con `@numba.jit(parallel=True, nopython=True)` utilizando `prange` para distribuir filas entre los núcleos del nodo.
2. **Medición de Escalamiento:** Medir Speedup y eficiencia para 1, 2, 4, 8 y 16 hilos en Kabré.
3. **Tarde (GPU):** Portar el tensor de estado a `cupy.ndarray` y escribir el **Kernel CUDA 2D Stencil** para evaluar simultáneamente todas las celdas en miles de hilos GPU.